In [513]:
import cadquery as cq
from jupyter_cadquery.cadquery import show
from jupyter_cadquery import set_sidecar

set_sidecar("Test", init=True)
objects = []

def show_object(obj):
    global objects
    objects.append(obj)
    show(*objects)

In [2]:
fuselage_step = cq.importers.importStep('fuselage_inlets.step')

In [3]:
fuselage_step

In [564]:
from OCP.BRepOffsetAPI import BRepOffsetAPI_MakeOffsetShape

In [25]:
sol = fuselage_step.findSolid()
sol = cq.Shape(sol.wrapped).scale(40)
sol = cq.Solid(sol.wrapped)
sol

In [565]:
def makeOffsetShape(shape, offset):
    maker = BRepOffsetAPI_MakeOffsetShape()
    maker.PerformBySimple(shape.wrapped, offset)
    of_shape = maker.Shape()
    return cq.Solid(of_shape)


In [566]:
def offset3D(self: cq.Workplane, offset: float) -> cq.Workplane:
    solid = self.findSolid()
    maker = BRepOffsetAPI_MakeOffsetShape()
    maker.PerformBySimple(solid.wrapped, offset)
    of_shape = maker.Shape()
    new_solid = cq.Solid(of_shape)
    return self.newObject([new_solid])

cq.Workplane.offset3D = offset3D
    

In [30]:
ss = cq.Workplane(sol).faces("<X").workplane(-100).split(keepTop=True, keepBottom=True)
ss

100% ⋮————————————————————————————————————————————————————————————⋮ (2/2)  8.51s


In [9]:
ss.last().offset3D(0.01)

In [10]:
ss

100% ⋮————————————————————————————————————————————————————————————⋮ (2/2)  0.01s


In [ ]:
os = makeOffsetShape(ss.last().findSolid(), -0.0004)
show(*[os, ss])

In [ ]:
ss.last().cut(os)

In [ ]:
line = cq.Workplane("XY").move(0,0).line(1,0)

In [ ]:
show(*[line, os])

In [ ]:
edge = line.wire().val()
type(edge.wrapped)

In [ ]:
#solid = ss.last().findSolid().wrapped
solid = ss.last().faces(">Z").first().findFace().wrapped
type(solid)
#solid


In [ ]:
from OCP.BRepProj import BRepProj_Projection
from OCP.gp import gp_Dir
from OCP.TopoDS import TopoDS_Shape

In [ ]:
ok = BRepProj_Projection(edge.wrapped, solid,  gp_Dir(0,1,1)).Shape()

In [ ]:
proj = cq.Edge(ok)

In [ ]:
show(*[line, os, proj])

In [ ]:
cq.Workplane(proj).edges().all()[3]

In [48]:
faces = ss.last().faces().val()

In [50]:
type(faces)

cadquery.occ_impl.shapes.Face

In [525]:
def airfoil(self: cq.Workplane, selig_file: str, chord: float, forConstruction: bool = False):
    file = open(selig_file, "r")
    point_list = []
    for line_num, line in enumerate(file):
        line: str = line
        if line_num < 1:
            pass
        else:
            tokens = [n for n in line.strip().split(" ") if n != ""]
            x = float(tokens[0])
            y = float(tokens[1])
            point_list.append((x * chord, y * chord))
    file.close()
    nose_point = min(point_list, key = lambda t: t[0])
    plane = self.plane
    new_plane = cq.Plane(xDir=plane.xDir, origin=(0,0,0), normal=plane.zDir)
    shape = (cq.Workplane(inPlane=new_plane).center(-nose_point[0], -nose_point[1])
             .splineApprox(points=point_list, forConstruction=forConstruction, tol=1e-3).close()
             .val())
    trans_shape = shape.translate(plane.origin)

    return self.newObject([trans_shape]).toPending()

cq.Workplane.airfoil = airfoil


In [560]:
from typing import Literal

def wing_root_segment(self: cq.Workplane, root_airfoil: str,  
                 root_chord: float, tip_chord: float, length: float, 
                 sweep: float=0, sweep_mode: Literal["distance", "angle"]="distance",
                 root_incidence: float=0, tip_incidence: float=0,
                 root_dihedral: float=0, tip_dihedral: float=0, tip_airfoil: str=None):
    tip_airfoil = tip_airfoil if tip_airfoil is not None else root_airfoil
    
    root_plane = self.plane.rotated((root_dihedral,0,-root_incidence))
    airfoil_root = (cq.Workplane(root_plane).airfoil(root_airfoil, root_chord))
    tip_plane = (airfoil_root.workplane(offset=-length, origin=(sweep,0,0))
                 .plane.rotated((tip_dihedral,0,-tip_incidence)))
    airfoil_tip = (airfoil_root.copyWorkplane(cq.Workplane(tip_plane))
                    .airfoil(tip_airfoil, tip_chord))
    wing = airfoil_tip.loft()  # ruled=True --> airfoils must have same number of points    
    return wing.newObject([tip_plane.location, wing.findSolid()])
    
cq.Workplane.wing_root_segment = wing_root_segment

In [561]:
def wing_segment(self: cq.Workplane, tip_airfoil: str, tip_chord: float, length: float, 
                 sweep: float=0, sweep_mode: Literal["distance", "angle"]="distance",
                 tip_incidence: float=0,tip_dihedral: float=0):
    
    airfoil_root = self.wires('>Z')
    airfoil_root.ctx.pendingWires = [airfoil_root.wire().val()]
    root_plane = airfoil_root.plane
    tip_origin = root_plane.origin
    tip_origin.x = tip_origin.x + sweep

    tip_plane = (airfoil_root.workplane(offset=-length, origin=tip_origin)
                 .plane.rotated((tip_dihedral,0,-tip_incidence)))
    airfoil_tip = (airfoil_root.copyWorkplane(cq.Workplane(tip_plane))
                    .airfoil(tip_airfoil, tip_chord))
    _wing = airfoil_tip.loft()  # ruled=True --> airfoils must have same number of points
    
    return _wing.newObject([_wing.findSolid()])
    
cq.Workplane.wing_segment = wing_segment

In [575]:
wing = (cq.Workplane('XZ').wing_root_segment(root_airfoil="./naca23013.5c.dat", root_chord=200, tip_chord=100, length=300, 
                           root_dihedral=5, tip_dihedral=10, sweep=100))
wing
wing = wing.wing_segment(tip_airfoil="./naca23013.5c.dat", tip_chord=50, length=200, tip_dihedral=0, sweep=50)
wing

In [578]:
wing.offset3D(-.8).sewAndFix()


AttributeError: 'Workplane' object has no attribute 'sewAndFix'